# 03 - Extra models and AMBER (Google Colab, GPU)

**GPU REQUIRED - RUN IN COLAB.** Extracts the same features as notebook 01 (last prompt token, every layer, yes/no scores) for
*other models* and for the *AMBER* benchmark, so the probes can be compared across models and tested for POPE -> AMBER transfer.
Everything calls `src/extract_features_multi.py` (unit-tested logic); only the model loading is GPU-only.

**Suggested order** (each run is one notebook run; change `MODEL_KEY` / `DATASET` and re-run from the top):
1. `qwen2vl-2b` + `amber`  (team model on AMBER; **required** for the transfer test)
2. `smolvlm-500m` + `pope`, then `smolvlm-500m` + `amber`  (cheapest, different architecture)
3. `qwen25vl-3b` + `pope` / `amber`  (same family, larger)
4. `llava15-7b` + `pope` / `amber`  (stretch; 4-bit, slowest)

Always run the **smoke test** cell first: it prints the yes/no token ids, `answer_mass` (must be close to 1) and s/question,
and estimates the total time. Do not start a multi-hour run if it prints a warning. Finished 100-question shards are skipped
on restart, so after a disconnect just re-run all cells with the same config.

In [ ]:
# ===================== CONFIG (edit only this cell) =====================
MODEL_KEY = "smolvlm-500m"      # qwen2vl-2b | qwen25vl-3b | smolvlm-500m | llava15-7b   (see the --list-models output below)
DATASET = "pope"                # "pope" | "amber"
POPE_SUBSET_SIZE = 3000         # first N rows of the seeded order (None = all 9000). Same rows for every model -> paired comparison
AMBER_PER_SUBSET = 1000         # random queries per AMBER subset (existence / attribute / relation)
LAYERS = "all"                  # "all" | "every:4" | "0,8,16,-1"
REPO_URL = "https://github.com/RUANLASS/SC4001Assignment_2026.git"
BRANCH = "extra-models-amber"   # set to "main" once merged
USE_DRIVE = True                # outputs go to Google Drive (survives disconnects)
DRIVE_DIR = "/content/drive/MyDrive/sc4001_extra"
AMBER_HF_IMAGES = None          # fallback if the Google-Drive image download fails: "visual-preference/AMBER"
# ========================================================================

## a. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "nvidia-smi not found")
if not Path("src/extract_features_multi.py").exists():
    target = Path("/content/SC4001Assignment_2026")
    if not target.exists():
        r = subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(target)], capture_output=True, text=True)
        print(r.stdout, r.stderr)
    assert (target / "src" / "extract_features_multi.py").exists(), "clone failed (private repo? wrong BRANCH?): upload the repo folder instead"
    os.chdir(target)
print("working dir:", os.getcwd())

In [ ]:
!pip install -q -r requirements-extra.txt
import torch; print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU, then re-run"
OUT_DIR = "results/cached_features"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive"); OUT_DIR = DRIVE_DIR; os.makedirs(OUT_DIR, exist_ok=True)
print("outputs ->", OUT_DIR)
!python src/extract_features_multi.py --list-models

## b. AMBER data (skip for POPE)
Metadata and ground truth come from the AMBER GitHub repo; pixels from the Google-Drive zip linked in its README
(file id below, taken from that README). If `gdown` hits a download quota, set `AMBER_HF_IMAGES = "visual-preference/AMBER"` in the
config and skip the zip, or download the zip by hand and unzip into `data/amber_images/`.

In [ ]:
if DATASET == "amber":
    if not Path("data/amber_repo").exists():
        !git clone -q https://github.com/junyangwang0410/AMBER data/amber_repo
    have_images = Path("data/amber_images").exists() and any(Path("data/amber_images").rglob("AMBER_*.jpg"))
    if AMBER_HF_IMAGES is None and not have_images:
        import gdown
        gdown.download(id="1MaCHgtupcZUjf007anNl4_MV0o4DjXvl", output="data/amber_images.zip", quiet=False)
        !unzip -q -o data/amber_images.zip -d data/amber_images
    AMBER_ARGS = f"--amber-hf-images {AMBER_HF_IMAGES}" if AMBER_HF_IMAGES else ""
    # Prints the label distribution per subset. EXPECTED (checked against the real annotations.json): 14,216 queries =
    # existence 4,924 (2,462 yes / 2,462 no) + attribute 7,628 (1,352 yes / 6,276 no) + relation 1,664 (975 yes / 689 no).
    # If this crashes with a message about 'truth', annotations.json has a different schema: fix build_amber_meta() in src/vlm_datasets.py.
    !python src/extract_features_multi.py --dataset amber --inspect --per-subset {AMBER_PER_SUBSET} {AMBER_ARGS}
else:
    AMBER_ARGS = ""


## c. Smoke test (20 questions, writes nothing)
Check: (1) the yes/no token ids look right, (2) `mean answer_mass` is close to 1 (e.g. > 0.9), (3) the prompt tokens count is sane,
(4) the time estimate fits your Colab budget. If the estimate is too long, lower `POPE_SUBSET_SIZE` / `AMBER_PER_SUBSET`.

In [ ]:
SIZE_ARGS = f"--subset-size {POPE_SUBSET_SIZE}" if (DATASET == "pope" and POPE_SUBSET_SIZE) else ""
COMMON = f"--model {MODEL_KEY} --dataset {DATASET} {SIZE_ARGS} --per-subset {AMBER_PER_SUBSET} --layers {LAYERS} --out-dir {OUT_DIR} {AMBER_ARGS}"
!python src/extract_features_multi.py {COMMON} --smoke 20

## d. Full run (resumable; checkpoints every 100 questions)

In [ ]:
!python src/extract_features_multi.py {COMMON}

## e. Bring the files back
Outputs: `{dataset}_{model}_features.pt` (**do not commit**, gitignored), `{dataset}_{model}_metadata.parquet`,
`{dataset}_{model}_run_config.json`. Copy them from Drive into the local repo's `results/cached_features/`.
Then run the CPU evaluation locally: `python src/probe_transfer_eval.py --models qwen2vl-2b smolvlm-500m --out-dir results/extra_models`.
The final verification printout above must show: 0 missing, 0 duplicates, 0 non-finite, `answer_mass` mean close to 1, and no `!!! FLAG` line.

In [ ]:
!ls -la {OUT_DIR}